# Indice

# Carga de las librerias

In [47]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sys
import os

sys.path.append(os.path.abspath(".."))

In [ ]:
# Ruta al dataset (ajusta si es necesario)
DATA_PATH = r'..\data\online_retail_II.xlsx'

# Si el Excel tiene varias hojas, podemos cargar la primera o especificar por nombre
#df = pd.read_excel(DATA_PATH, sheet_name=1)
xl = pd.ExcelFile(DATA_PATH)
df = pd.concat([pd.read_excel(DATA_PATH, sheet_name=s) for s in xl.sheet_names], ignore_index=True)
print(f'Filas: {len(df):,} | Columnas: {len(df.columns)}')

Filas: 1,067,371 | Columnas: 8


# Normalizacion de las columnas

In [49]:
# Creamos una copia para trabajar con ella y mantener el original intacto en caso de error
df_cleaning = df.copy()

def normalize_column_names(df):
    df_cleaned = df.copy()
    df_cleaned.columns = (
        df_cleaned.columns.str.lower()     # Minúsculas
        .str.strip()                       # Sin espacios externos
        .str.replace(" ", "_")             # Reemplazo por guiones bajos
        .str.replace('[^0-9a-zA-Z_]', '', regex=True)  # Elimina caracteres especiales
    )
    return df_cleaned

df_cleaning = normalize_column_names(df_cleaning)
df_cleaning.columns

Index(['invoice', 'stockcode', 'description', 'quantity', 'invoicedate',
       'price', 'customer_id', 'country'],
      dtype='object')

In [50]:
df_cleaning.rename(columns={'stockcode': 'stock_code', 'invoicedate': 'invoice_date'}, inplace=True)
df_cleaning.columns

Index(['invoice', 'stock_code', 'description', 'quantity', 'invoice_date',
       'price', 'customer_id', 'country'],
      dtype='object')

# Proceso de limpieza

## Eliminación de nulos

In [51]:
# Eliminamos los registros con ID nulo ya que para el modelado no nos sirven si no se pueden identificar los clientes
df_cleaning = df_cleaning.dropna(subset=["customer_id"])

In [52]:
# Eliminamos los duplicados reales
df_cleaning = df_cleaning.drop_duplicates()

In [53]:
# La mayoria de registros con precio = 0 no tenian id de cliente asociado, por eso sacamos los que hay actualmente para saber su cantidad exacta
print(f"Número de filas con precios a 0: {(df_cleaning["price"] == 0).sum()}")
df_cleaning[df_cleaning["price"] == 0]

Número de filas con precios a 0: 70


,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126.0,United Kingdom
6781,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.0,15658.0,United Kingdom
16107,490727,M,Manual,1,2009-12-07 16:38:00,0.0,17231.0,United Kingdom
18738,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,2009-12-08 15:25:00,0.0,14108.0,United Kingdom
18739,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,2009-12-08 15:25:00,0.0,14108.0,United Kingdom
...,...,...,...,...,...,...,...,...
1004540,577129,22464,HANGING METAL HEART LANTERN,4,2011-11-17 19:52:00,0.0,15602.0,United Kingdom
1005014,577168,M,Manual,1,2011-11-18 10:42:00,0.0,12603.0,Germany
1006110,577314,23407,SET OF 2 TRAYS HOME SWEET HOME,2,2011-11-18 13:23:00,0.0,12444.0,Norway
1011446,577696,M,Manual,1,2011-11-21 11:57:00,0.0,16406.0,United Kingdom


In [54]:
# Al ser unicamente 70 registros, los eliminamos
df_cleaning = df_cleaning[df_cleaning["price"] > 0]

# Creación de variables

In [55]:
df_cleaning["is_return"] = df_cleaning["quantity"] < 0

df_cleaning["quantity_pos"] = df_cleaning["quantity"].clip(lower=0)
df_cleaning["quantity_neg"] = df_cleaning["quantity"].clip(upper=0)

df_cleaning["total_price"] = df_cleaning["quantity"] * df_cleaning["price"]
df_cleaning.columns

Index(['invoice', 'stock_code', 'description', 'quantity', 'invoice_date',
       'price', 'customer_id', 'country', 'is_return', 'quantity_pos',
       'quantity_neg', 'total_price'],
      dtype='object')

In [56]:
df_cleaning[df_cleaning["quantity"] < 0].head(5)

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,is_return,quantity_pos,quantity_neg,total_price
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,True,0,-12,-35.4
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia,True,0,-6,-9.9
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia,True,0,-4,-17.0
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia,True,0,-6,-12.6
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,True,0,-12,-35.4


In [57]:
df_cleaning[df_cleaning["quantity"] > 0].head(5)

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,is_return,quantity_pos,quantity_neg,total_price
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,12,0,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,12,0,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,12,0,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,48,0,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,24,0,30.0


In [58]:
df_cleaning["year"] = df_cleaning["invoice_date"].dt.year
df_cleaning["month"] = df_cleaning["invoice_date"].dt.month
df_cleaning["day"] = df_cleaning["invoice_date"].dt.day
df_cleaning["day_of_week"] = df_cleaning["invoice_date"].dt.dayofweek
df_cleaning["hour"] = df_cleaning["invoice_date"].dt.hour

In [59]:
df_cleaning.tail(5)

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,is_return,quantity_pos,quantity_neg,total_price,year,month,day,day_of_week,hour
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,False,6,0,12.60,2011,12,9,4,12
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,False,4,0,16.60,2011,12,9,4,12
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,False,4,0,16.60,2011,12,9,4,12
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France,False,3,0,14.85,2011,12,9,4,12
1067370,581587,POST,POSTAGE,1,2011-12-09 12:50:00,18.00,12680.0,France,False,1,0,18.00,2011,12,9,4,12


In [60]:
df_cleaning["invoice_date"].dt.hour.value_counts().sort_index()


invoice_date
6         41
7       1113
8      15473
9      41947
10     74212
11     99306
12    141468
13    129271
14    108621
15     87904
16     52769
17     27561
18      7712
19      8410
20      2006
21         1
Name: count, dtype: int64

In [61]:
df_cleaning.shape

(797815, 17)

In [67]:
print(df_cleaning.shape)
print(df_cleaning.isna().sum())

(797815, 17)
invoice         0
stock_code      0
description     0
quantity        0
invoice_date    0
price           0
customer_id     0
country         0
is_return       0
quantity_pos    0
quantity_neg    0
total_price     0
year            0
month           0
day             0
day_of_week     0
hour            0
dtype: int64


# Eliminación de espacios y errores en los registros de texto

In [66]:
# columnas de texto que sí quieres limpiar
text_cols = ["description", "country"]

for col in text_cols:
    df_cleaning[col] = (
        df_cleaning[col]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

code_cols = ["invoice", "stock_code"]

for col in code_cols:
    df_cleaning[col] = (
        df_cleaning[col]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", "", regex=True)
    )

In [63]:
df_cleaning.head(5)

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,is_return,quantity_pos,quantity_neg,total_price,year,month,day,day_of_week,hour
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,12,0,83.4,2009,12,1,1,7
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,12,0,81.0,2009,12,1,1,7
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,12,0,81.0,2009,12,1,1,7
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,48,0,100.8,2009,12,1,1,7
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,24,0,30.0,2009,12,1,1,7


# Modificación de los tipos de las columnas

In [68]:
df_cleaning.dtypes

invoice         string[python]
stock_code      string[python]
description     string[python]
quantity                 int64
invoice_date    datetime64[ns]
price                  float64
customer_id            float64
country         string[python]
is_return                 bool
quantity_pos             int64
quantity_neg             int64
total_price            float64
year                     int32
month                    int32
day                      int32
day_of_week              int32
hour                     int32
dtype: object

In [69]:
df_cleaning["customer_id"] = df_cleaning["customer_id"].astype("Int64")
df_cleaning["country"] = df_cleaning["country"].astype("category")

In [70]:
df_cleaning.dtypes

invoice         string[python]
stock_code      string[python]
description     string[python]
quantity                 int64
invoice_date    datetime64[ns]
price                  float64
customer_id              Int64
country               category
is_return                 bool
quantity_pos             int64
quantity_neg             int64
total_price            float64
year                     int32
month                    int32
day                      int32
day_of_week              int32
hour                     int32
dtype: object

In [71]:
df_cleaning.head()

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,is_return,quantity_pos,quantity_neg,total_price,year,month,day,day_of_week,hour
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,False,12,0,83.4,2009,12,1,1,7
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,12,0,81.0,2009,12,1,1,7
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,12,0,81.0,2009,12,1,1,7
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,False,48,0,100.8,2009,12,1,1,7
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,False,24,0,30.0,2009,12,1,1,7


# Exportación a un parquet

In [73]:
df_cleaning.to_parquet("..\data\data_clean.parquet", index=False)

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\User\AppData\Local\Temp\ipykernel_23120\2662599555.py:1: SyntaxWarning: invalid escape sequence '\d'
  df_cleaning.to_parquet("..\data\data_clean.parquet", index=False)
